# SPW quickstart: a weight-perturbed GraphCast ensemble, scored with SwissClim

A small version of the paper's experiments
([Stochastically Perturbed Weights](https://arxiv.org/abs/2609.08412)): GraphCast with the production
perturbation `graphcast_all` (Gaussian noise with sigma 0.01 on all weights, member k seeded with 42 + k),
initialised 2023-01-02 00 UTC, 2 members, 24 h. The ensemble is then scored against ERA5 with
[SwissClim Evaluations](https://github.com/swiss-ai/SwissClim_Evaluations): CRPS and the RMSE of the
ensemble mean.

Needs Linux, one NVIDIA GPU with a CUDA 13 driver (JAX) and internet: the initial conditions and the
truth come from ARCO-ERA5 and the model weights from the public GraphCast bucket, all on Google Cloud.
Tested on a GH200 (96 GB). On CSCS Clariden use [spw_clariden.ipynb](spw_clariden.ipynb) instead, which
reads only local data and runs in the production containers.

## Setup (once, in a terminal)

The pins and the override file reproduce the production container
([containers/graphcast.dockerfile](../containers/graphcast.dockerfile)).

```bash
git clone https://github.com/MeteoSwiss/ai-models-ensembles.git && cd ai-models-ensembles
uv venv --python 3.12 .venv && source .venv/bin/activate
uv pip install torch   # the CUDA build for your machine, see https://pytorch.org/get-started/locally/
uv pip install -e ".[graphcast]" matplotlib ipykernel --override env/graphcast-overrides.txt
python -m ipykernel install --user --name spw-quickstart
```

Then open this notebook with the `spw-quickstart` kernel.

In [ ]:
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import xarray as xr
import yaml

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUT = REPO / "quickstart_output"
OUT.mkdir(exist_ok=True)
FORECAST = OUT / "forecast.zarr"
INIT = "2023-01-02T00:00"
VARIABLES = [
    "10m_u_component_of_wind",
    "10m_v_component_of_wind",
    "2m_temperature",
    "geopotential",
    "mean_sea_level_pressure",
    "specific_humidity",
    "temperature",
    "u_component_of_wind",
    "v_component_of_wind",
]
# ERA5 on Google Cloud (ARCO): the source of the initial conditions and of the truth
ERA5 = "gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3"

## 1. Inference

The CLI does per init what the production submitter (`scripts/submit_all_inference.sh`) does for the
whole grid, here with 2 members and 24 h. The first run downloads the GraphCast weights (1.2 GB) into
`~/.cache/earth2studio`.

In [ ]:
cmd = [
    sys.executable,
    "-m",
    "ai_models_ensembles.cli",
    "infer",
    "--model",
    "graphcast_operational",
    "--init",
    INIT,
    "--lead-hours",
    "24",
    "--members",
    "2",
    "--weight-magnitude",
    "0.01",
    "--layer",
    "all",
    "--data-source",
    "arco",
    "--seed",
    "42",
    "--output-levels",
    "500,850",
    "--output-vars",
    ",".join(VARIABLES),
    "--output",
    str(FORECAST),
]
if not FORECAST.exists():
    subprocess.run(cmd, check=True, cwd=REPO)
fc = xr.open_zarr(FORECAST)
fc

In [ ]:
t2m = fc["2m_temperature"].isel(init_time=0).sel(lead_time=pd.Timedelta(hours=24))
fig, axes = plt.subplots(1, 2, figsize=(13, 3.6))
t2m.sel(ensemble=0).plot(ax=axes[0], cmap="RdYlBu_r")
axes[0].set_title("2m temperature, member 0, +24 h")
(t2m.sel(ensemble=0) - t2m.sel(ensemble=1)).plot(ax=axes[1], cmap="RdBu_r", robust=True)
axes[1].set_title("member 0 - member 1 (spread from the weight noise)")
fig.tight_layout()

## 2. Evaluation with SwissClim

The same configuration as the production evaluations (`scripts/evaluate_baselines.sh`), reduced to two
modules, one init and 24 h. The truth is copied first: only the 5 valid times, 5 variables and 2 levels
that are scored (about 150 MB), because SwissClim reading the full ERA5 store over the network takes hours.

In [ ]:
TRUTH = OUT / "era5_truth.zarr"
if not TRUTH.exists():
    era5 = xr.open_zarr(ERA5, storage_options={"token": "anon"})
    times = pd.date_range("2023-01-02T00", "2023-01-03T00", freq="6h")
    surface = ["2m_temperature", "mean_sea_level_pressure", "10m_u_component_of_wind"]
    upper = ["geopotential", "temperature"]
    sub = xr.merge(
        [era5[surface].sel(time=times), era5[upper].sel(time=times, level=[500, 850])]
    ).load()
    for v in sub.variables.values():
        v.encoding = {}
    sub.to_zarr(TRUTH, mode="w")
xr.open_zarr(TRUTH)

In [ ]:
modules = [
    "maps",
    "histograms",
    "wd_kde",
    "energy_spectra",
    "vertical_profiles",
    "deterministic",
    "ets",
    "fss",
    "probabilistic",
    "ssim",
    "multivariate",
]
config = {
    "paths": {"ml": [str(FORECAST)], "nwp": [str(TRUTH)], "output_root": str(OUT / "eval")},
    "selection": {
        "levels": [500, 850],
        "temporal_resolution_hours": None,
        "datetimes": ["2023-01-02T00:00:2023-01-03T00"],
        "latitudes": [90.0, -89.75],
        "longitudes": [0.0, 359.75],
        "variables_2d": ["2m_temperature", "mean_sea_level_pressure", "10m_u_component_of_wind"],
        "variables_3d": ["geopotential", "temperature"],
        "ensemble_members": None,
        "ensemble": {"deterministic": "mean", "probabilistic": "prob"},
        "check_missing": False,
    },
    "plotting": {
        "random_seed": 42,
        "plot_datetime": None,
        "plot_ensemble_members": [0],
        "dpi": 48,
        "output_mode": "both",
        "probabilistic_line_plots": True,
    },
    "modules": {m: m in ("deterministic", "probabilistic") for m in modules},
    "lead_time": {
        "mode": "stride",
        "stride": {"hours": 6},
        "max_hour": 24,
        "panel": {"strategy": "evenly_spaced", "count": 5},
    },
    "metrics": {
        "deterministic": {
            "include": ["MAE", "RMSE"],
            "standardized_include": ["MAE", "RMSE"],
            "report_per_level": True,
            "error_maps": False,
        }
    },
    "performance": {"dask_scheduler": "distributed", "dask_profile": "balanced"},
}
cfg = OUT / "eval_config.yaml"
cfg.write_text(yaml.safe_dump(config, sort_keys=False))
subprocess.run(
    [sys.executable, "-m", "swissclim_evaluations.cli", "--config", str(cfg)], check=True
)

In [ ]:
ev = OUT / "eval"
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, var, unit in [(axes[0], "2m_temperature", "K"), (axes[1], "geopotential_500", "m2/s2")]:
    crps = pd.read_csv(ev / "probabilistic" / f"crps_line_{var}_by_lead_ensprob.csv")
    rmse = pd.read_csv(next((ev / "deterministic").glob(f"det_line_{var}_RMSE_by_lead_*.csv")))
    ax.plot(crps.lead_time_hours, crps.CRPS, marker="o", label="CRPS")
    ax.plot(rmse.lead_time_hours, rmse.RMSE, marker="o", label="RMSE of the ensemble mean")
    ax.set(xlabel="lead time (h)", ylabel=unit, title=var)
    ax.legend()
fig.tight_layout()

## Next steps

- What each experiment tests and the production settings: [docs/EXPERIMENTS.md](../docs/EXPERIMENTS.md).
- Every figure and table of the paper: [docs/REPRODUCING.md](../docs/REPRODUCING.md).
- Other models and perturbations: `python -m ai_models_ensembles.cli infer --help`; the production
  flags of each experiment are in `scripts/submit_all_inference.sh`.